# Customer Churn Over Time

In this notebook, we performed a series of transforamtion steps to determine customer churn status for each customer on a monthly basis, including:
1) [Load and Preprocess Data to include Appropriate Columns](#load_process)
2) [Compute the Month-end Date for Each Month of Year](#month-end)
3) [Merge Month-end Dates into the Churn Table](#merge)
4) [Identify the Latest Purchase Time for Each Customer Per Month](#latest-purchase)
5) [Calculate the Interval Days between each Month-end and corresponding Latest Purchase Time](#interval)
6) [Use the Interval Days and Churn Threshold to determine each customer's churn status for every month](#churn-label)
7) [Export to CSV](#csv)

In [1]:
import pandas as pd
import warnings
import json
import numpy as np
from datetime import datetime, date, timedelta

In [2]:
# Ignore all warnings in output
warnings.filterwarnings('ignore')

In [3]:
# Set the maximum column width for pandas DataFrame display
pd.set_option('max_colwidth', 2000)

<a name="load_process"></a>
### Load and Preprocess Data

In [4]:
# Load the CSV file into a pandas DataFrame and 
# parse the 'order_time' column as datetime
df = pd.read_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_full_mx.csv',
    parse_dates=['order_time'],
)

In [5]:
# Extract necessary columns and remove duplicates
df_custchurn = df[
    ['order_time', 'customer_id', 'churn_threshold']
].drop_duplicates().copy()

In [6]:
# Print the shape and information of the extracted DataFrame
print(df_custchurn.shape)
print(df_custchurn.info())

(34831, 3)
<class 'pandas.core.frame.DataFrame'>
Int64Index: 34831 entries, 0 to 488453
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_time       34831 non-null  datetime64[ns]
 1   customer_id      34831 non-null  int64         
 2   churn_threshold  34831 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB
None


In [7]:
df_custchurn.head()

,order_time,customer_id,churn_threshold
0,2021-04-14 06:15:00,517,44.242406
11,2024-03-05 10:17:23,519,18.650711
31,2024-03-01 11:40:58,519,18.650711
51,2023-12-05 08:49:24,519,18.650711
71,2021-06-20 12:11:09,776,98.853427


<a name="month-end"></a>
### Compute the Month-end Date for Each Month of Year

In [8]:
start_date = df_custchurn['order_time'].min().date()
end_date = df_custchurn['order_time'].max().date()
df_dates = pd.DataFrame(
    {"Dates": pd.date_range(start_date, end_date)}
)

In [9]:
df_dates.head()

,Dates
0,2019-05-05
1,2019-05-06
2,2019-05-07
3,2019-05-08
4,2019-05-09


In [10]:
# Extract year and month from 'Dates' column
df_dates['year'] = df_dates['Dates'].dt.year
df_dates['month'] = df_dates['Dates'].dt.month

In [11]:
df_dates.head()

,Dates,year,month
0,2019-05-05,2019,5
1,2019-05-06,2019,5
2,2019-05-07,2019,5
3,2019-05-08,2019,5
4,2019-05-09,2019,5


In [12]:
latest_date = df_dates['Dates'].max()

def calc_monthend(x):
    if (x['year'] == latest_date.year) and (x['month'] == latest_date.month):
        return latest_date + timedelta(days=1)
    else:
        if x['month'] < 12:
            return date(x['year'], x['month'] + 1, 1)
        else:
            return date(x['year'] + 1, 1, 1)
    
df_dates['month_end'] = df_dates.apply(calc_monthend, axis=1)

In [13]:
df_dates = df_dates[['year', 'month', 'month_end']].drop_duplicates()
df_dates

,year,month,month_end
0,2019,5,2019-06-01
27,2019,6,2019-07-01
57,2019,7,2019-08-01
88,2019,8,2019-09-01
119,2019,9,2019-10-01
149,2019,10,2019-11-01
180,2019,11,2019-12-01
210,2019,12,2020-01-01
241,2020,1,2020-02-01
272,2020,2,2020-03-01


<a name="merge"></a>
### Merge Month-end Dates into the Churn Table

In [14]:
# Perform a cross join between df_custchurn and the month DataFrame
df_churn = df_custchurn.merge(
    df_dates,
    how='cross',
)

In [15]:
df_churn.head()

,order_time,customer_id,churn_threshold,year,month,month_end
0,2021-04-14 06:15:00,517,44.242406,2019,5,2019-06-01
1,2021-04-14 06:15:00,517,44.242406,2019,6,2019-07-01
2,2021-04-14 06:15:00,517,44.242406,2019,7,2019-08-01
3,2021-04-14 06:15:00,517,44.242406,2019,8,2019-09-01
4,2021-04-14 06:15:00,517,44.242406,2019,9,2019-10-01


<a name="latest-purchase"></a>
### Identify the Latest Purchase Time for Each Customer Per Month

In [16]:
# Filter the churn data for records where the order time 
# is earlier than the month-end
df_churn = df_churn[
    df_churn['order_time'] < df_churn['month_end']
]

In [17]:
# Sort and reset the index for clean display
df_churn = df_churn[
    [
        'month_end', 'customer_id', 'order_time', 
        'year', 'month', 'churn_threshold'
    ]
].sort_values(
    ['month_end', 'customer_id', 'order_time'],
    ascending=[True, True, True],
).reset_index(drop=True)

df_churn.head(10)

,month_end,customer_id,order_time,year,month,churn_threshold
0,2019-06-01,4,2019-05-05 22:20:05,2019,5,1.154701
1,2019-06-01,4,2019-05-06 06:00:00,2019,5,1.154701
2,2019-06-01,4,2019-05-06 06:15:00,2019,5,1.154701
3,2019-06-01,5,2019-05-06 11:00:02,2019,5,159.941811
4,2019-06-01,5,2019-05-06 11:01:43,2019,5,159.941811
5,2019-06-01,5,2019-05-06 11:01:47,2019,5,159.941811
6,2019-06-01,5,2019-05-06 11:15:00,2019,5,159.941811
7,2019-06-01,5,2019-05-06 11:45:00,2019,5,159.941811
8,2019-06-01,5,2019-05-06 16:07:00,2019,5,159.941811
9,2019-06-01,5,2019-05-06 17:03:11,2019,5,159.941811


In [18]:
# Add a column for the latest order time per customer for each month
df_churn['latest_order_time'] = (
    df_churn.groupby(
        ['month_end', 'customer_id']
    )
    ['order_time'].transform(np.max)
)

df_churn.head()

,month_end,customer_id,order_time,year,month,churn_threshold,latest_order_time
0,2019-06-01,4,2019-05-05 22:20:05,2019,5,1.154701,2019-05-06 06:15:00
1,2019-06-01,4,2019-05-06 06:00:00,2019,5,1.154701,2019-05-06 06:15:00
2,2019-06-01,4,2019-05-06 06:15:00,2019,5,1.154701,2019-05-06 06:15:00
3,2019-06-01,5,2019-05-06 11:00:02,2019,5,159.941811,2019-05-31 14:15:00
4,2019-06-01,5,2019-05-06 11:01:43,2019,5,159.941811,2019-05-31 14:15:00


In [19]:
# Drop the original 'order_time' column and remove duplicates
df_churn = df_churn.drop(
    'order_time', axis=1
).drop_duplicates()

In [20]:
df_churn.head(10)

,month_end,customer_id,year,month,churn_threshold,latest_order_time
0,2019-06-01,4,2019,5,1.154701,2019-05-06 06:15:00
3,2019-06-01,5,2019,5,159.941811,2019-05-31 14:15:00
44,2019-06-01,6,2019,5,2158.089896,2019-05-21 15:04:50
46,2019-07-01,4,2019,6,1.154701,2019-05-06 06:15:00
49,2019-07-01,5,2019,6,159.941811,2019-05-31 14:15:00
90,2019-07-01,6,2019,6,2158.089896,2019-05-21 15:04:50
92,2019-07-01,8,2019,6,62.844198,2019-06-29 10:34:27
94,2019-07-01,9,2019,6,6.703924,2019-06-26 10:01:53
95,2019-07-01,13,2019,6,44.810789,2019-06-29 08:21:06
97,2019-07-01,18,2019,6,102.910747,2019-06-25 08:00:00


<a name="interval"></a>
### Calculate Interval Days

In [21]:
# Define a function to compute the number of days between 
# the latest purchase and the month-end
def compute_interval(x):
    return (
        x['month_end'].date() - x['latest_order_time'].date()
    ).days
    
# Apply the function to compute the interval between the 
# latest purchase and the month-end
df_churn['interval_to_latest_purchase'] = (
    df_churn.apply(compute_interval, axis=1)
)

In [22]:
df_churn.head(10)

,month_end,customer_id,year,month,churn_threshold,latest_order_time,interval_to_latest_purchase
0,2019-06-01,4,2019,5,1.154701,2019-05-06 06:15:00,26
3,2019-06-01,5,2019,5,159.941811,2019-05-31 14:15:00,1
44,2019-06-01,6,2019,5,2158.089896,2019-05-21 15:04:50,11
46,2019-07-01,4,2019,6,1.154701,2019-05-06 06:15:00,56
49,2019-07-01,5,2019,6,159.941811,2019-05-31 14:15:00,31
90,2019-07-01,6,2019,6,2158.089896,2019-05-21 15:04:50,41
92,2019-07-01,8,2019,6,62.844198,2019-06-29 10:34:27,2
94,2019-07-01,9,2019,6,6.703924,2019-06-26 10:01:53,5
95,2019-07-01,13,2019,6,44.810789,2019-06-29 08:21:06,2
97,2019-07-01,18,2019,6,102.910747,2019-06-25 08:00:00,6


<a name="churn-label"></a>
### Determine Customer Churn Status over Time

A customer will be classified as **CHURN** if the interval days exceed their churn threshold; otherwise, they will be identified as **NOT CHURN**.

In [23]:
# Determine churn status by checking if the interval 
# exceeds the churn threshold
df_churn['churn'] = (
    df_churn['interval_to_latest_purchase']
    > df_churn['churn_threshold']
)

In [24]:
# Display churn information for the first 20 customers on the latest date
df_churn[
    df_churn['month_end'] == latest_date + timedelta(days=1)
].head(20)

,month_end,customer_id,year,month,churn_threshold,latest_order_time,interval_to_latest_purchase,churn
774604,2024-03-07,4,2024,3,1.154701,2019-05-06 06:15:00,1767,True
774607,2024-03-07,5,2024,3,159.941811,2021-06-06 07:00:00,1005,True
774653,2024-03-07,6,2024,3,2158.089896,2023-08-08 09:59:31,212,False
774656,2024-03-07,8,2024,3,62.844198,2023-03-12 07:26:43,361,True
774810,2024-03-07,9,2024,3,6.703924,2024-03-05 10:36:33,2,False
775781,2024-03-07,10,2024,3,55.121945,2023-03-05 10:38:48,368,True
775847,2024-03-07,11,2024,3,102.910747,2021-09-22 08:05:46,897,True
775849,2024-03-07,12,2024,3,30.753554,2023-08-31 08:16:53,189,True
776176,2024-03-07,13,2024,3,44.810789,2023-12-09 07:39:53,89,True
776429,2024-03-07,14,2024,3,39.925480,2024-02-14 08:00:00,22,False


In [25]:
# Final clean-up: drop 'month_end', 
# sort by customer ID, year, and month, and reset index
df_churn_final = (
    df_churn.drop('month_end', axis=1)
    .sort_values(
        ['customer_id', 'year', 'month'],
        ascending=[True, True, True],
    ).reset_index(drop=True)
)

In [26]:
# Display the data for customer with ID 5
df_churn_final[
    df_churn_final['customer_id'] == 5
]

,customer_id,year,month,churn_threshold,latest_order_time,interval_to_latest_purchase,churn
59,5,2019,5,159.941811,2019-05-31 14:15:00,1,False
60,5,2019,6,159.941811,2019-05-31 14:15:00,31,False
61,5,2019,7,159.941811,2019-05-31 14:15:00,62,False
62,5,2019,8,159.941811,2019-08-03 11:01:49,29,False
63,5,2019,9,159.941811,2019-08-03 11:01:49,59,False
64,5,2019,10,159.941811,2019-08-03 11:01:49,90,False
65,5,2019,11,159.941811,2019-08-03 11:01:49,120,False
66,5,2019,12,159.941811,2019-12-15 15:05:24,17,False
67,5,2020,1,159.941811,2019-12-15 15:05:24,48,False
68,5,2020,2,159.941811,2019-12-15 15:05:24,77,False


<a name="csv"></a>
### Export to CSV

In [27]:
# Save the final churn data to a CSV file
df_churn_final.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_customer_churn_mx.csv',
    header=True,
    index=False,
)